# How Prediction Markets Converge to the Truth

Prediction markets price real-world events as probabilities. But how *good* are those prices?
This notebook uses the [ImpliedData](https://implieddata.com) sample — 1-hour candles for 130
top-volume Polymarket & Manifold markets — to look at three things:

1. **Price paths** — what a market's "opinion" looks like over time
2. **Convergence** — how prices approach the final outcome (0 or 1) as resolution nears
3. **The catalog** — what 230k+ prediction markets are actually about

*Data: ImpliedData (implieddata.com) — full dataset covers 230k+ markets, 19M+ trades,
intraday resolution, updated daily. This sample: 1h candles, 130 markets.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True

# Locate the data — works on Kaggle (any /kaggle/input mount) and locally.
import glob, os
_hits = glob.glob("/kaggle/input/*/sample_markets_lookup.parquet") + ["./sample_markets_lookup.parquet"]
DIR = os.path.dirname(next(p for p in _hits if os.path.exists(p))) or "."

poly   = pd.read_parquet(f"{DIR}/polymarket_ohlcv_1h_sample.parquet")
mani   = pd.read_parquet(f"{DIR}/manifold_ohlcv_1h_sample.parquet")
lookup = pd.read_parquet(f"{DIR}/sample_markets_lookup.parquet")
meta   = pd.read_parquet(f"{DIR}/markets_metadata.parquet")

candles = pd.concat([poly, mani], ignore_index=True)
candles["timestamp"] = pd.to_datetime(candles["timestamp"], utc=True)
df = candles.merge(lookup, on="market_id", suffixes=("", "_m"))
print(f"{df.market_id.nunique()} markets, {len(df):,} hourly candles")
df.head(3)

## 1. Price paths — markets thinking out loud

Each line is the hourly close (implied probability) of one market.

In [ ]:
# Most-traded markets in the sample
top = (df.groupby(["market_id", "title"])["volume"].sum()
         .sort_values(ascending=False).head(6).reset_index())

fig, ax = plt.subplots()
for _, row in top.iterrows():
    m = df[df.market_id == row.market_id].sort_values("timestamp")
    ax.plot(m.timestamp, m.close, label=str(row.title)[:48], linewidth=1.4)
ax.set_ylabel("Implied probability")
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc="best")
ax.set_title("Hourly implied probability — most-traded sample markets")
plt.tight_layout()

## 2. Convergence — do prices approach the truth?

For markets that have **resolved**, we know the answer (Yes = 1, No = 0).
We can measure how far the market price was from that final answer, as a function
of time remaining until resolution. A well-functioning market should converge:
the error should shrink as the event gets closer.

In [ ]:
resolved = lookup[(lookup.status == "resolved") & lookup.resolution.isin(["Yes", "No"])].copy()
resolved["outcome"] = (resolved.resolution == "Yes").astype(float)
print(f"{len(resolved)} resolved Yes/No markets in the sample")

r = df.merge(resolved[["market_id", "outcome"]], on="market_id")
r["closes_ts"] = pd.to_datetime(r["closes"], utc=True)
r["hours_to_resolution"] = (r.closes_ts - r.timestamp).dt.total_seconds() / 3600
r = r[r.hours_to_resolution.between(0, 24 * 14)]          # final two weeks
r["abs_error"] = (r.close - r.outcome).abs()

bins = [0, 6, 12, 24, 48, 96, 168, 336]
labels = ["<6h", "6-12h", "12-24h", "1-2d", "2-4d", "4-7d", "1-2w"]
r["bucket"] = pd.cut(r.hours_to_resolution, bins=bins, labels=labels)

conv = r.groupby("bucket", observed=True)["abs_error"].agg(["mean", "count"])
fig, ax = plt.subplots()
conv["mean"][::-1].plot(kind="bar", ax=ax, color="#2563eb")
ax.set_ylabel("Mean |price − outcome|")
ax.set_xlabel("Time before resolution")
ax.set_title("Markets get more accurate as resolution approaches")
plt.xticks(rotation=0); plt.tight_layout()
conv

The bar chart reads like a countdown: two weeks out, prices still carry real uncertainty;
in the final hours, the market has usually 'figured it out'. This is the core property
that makes prediction-market data valuable as a probability time series.

*(With only ~40 resolved markets in this free sample the buckets are noisy — the
[full dataset](https://implieddata.com) has tens of thousands of resolved markets
for proper calibration work.)*

## 3. What do 230k prediction markets ask about?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
meta.platform.value_counts().plot(kind="bar", ax=axes[0], color=["#2563eb", "#7c3aed"])
axes[0].set_title("Markets per platform"); axes[0].tick_params(rotation=0)
meta.status.value_counts().plot(kind="bar", ax=axes[1], color="#059669")
axes[1].set_title("Market status distribution"); axes[1].tick_params(rotation=0)
plt.tight_layout()

cats = meta[meta.category.notna() & (meta.category != "")].category.value_counts().head(12)
print(cats)

## Going further

Ideas this sample can't support but the full dataset can:

- **Calibration curves** ("when markets say 70%, does it happen 70% of the time?") across tens of thousands of resolved markets
- **Cross-platform comparison** — does real money (Polymarket) beat play money (Manifold)?
- **Event studies at 1-minute resolution** — how fast do markets absorb breaking news?
- **Volatility & volume microstructure** of event contracts

Full dataset — all markets, full history since 2020, 1m/5m/15m/1h/1d candles, daily updates:
**[implieddata.com](https://implieddata.com)**

*Sample license: CC BY-NC 4.0 (non-commercial research). Attribution: "Data: ImpliedData (implieddata.com)".*